In [2]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from scipy import stats
import statsmodels.api as sm

try:
    from google.colab import files
    files.upload()   # model_A / model_B / model_C 세 파일 모두 선택
except Exception:
    pass

PATH_A = "model_A_financial_final.csv"
PATH_B = "model_B_nonfinancial_final.csv"
PATH_C = "model_C_nonfinancial_new_final.csv"
N_BOOT = 1000

Saving model_A_financial_final.csv to model_A_financial_final (1).csv
Saving model_B_nonfinancial_final.csv to model_B_nonfinancial_final (1).csv
Saving model_C_nonfinancial_new_final.csv to model_C_nonfinancial_new_final.csv


In [3]:
a = pd.read_csv(PATH_A); b = pd.read_csv(PATH_B); c = pd.read_csv(PATH_C)

def clean(df):
    # 숫자형이 아닌 컬럼(문자열/카테고리, 예: AGE_BAND) 자동 제외 — string 타입도 잡음
    nonnum = [x for x in df.columns if not pd.api.types.is_numeric_dtype(df[x])]
    if nonnum: print("제외:", nonnum)
    df = df.drop(columns=nonnum)
    for x in df.columns:                        # bool 더미 → int
        if df[x].dtype == bool: df[x] = df[x].astype(int)
    return df
a, b, c = clean(a), clean(b), clean(c)

m = a.merge(b.drop(columns=["TARGET"]), on="SK_ID_CURR", how="inner") \
     .merge(c.drop(columns=["TARGET"]), on="SK_ID_CURR", how="inner")
y = m["TARGET"].astype(int).values
A_cols = [x for x in a.columns if x not in ("SK_ID_CURR","TARGET")]
B_cols = [x for x in b.columns if x not in ("SK_ID_CURR","TARGET")]
C_cols = [x for x in c.columns if x not in ("SK_ID_CURR","TARGET")]
grp = {**{x:"A" for x in A_cols}, **{x:"B" for x in B_cols}, **{x:"C" for x in C_cols}}
print(f"rows={len(m):,}  결측={m.isna().sum().sum()}  연체율={y.mean():.4f}")
print(f"A={len(A_cols)}  B={len(B_cols)}  C={len(C_cols)}  M4={len(A_cols+B_cols+C_cols)}")
assert len(A_cols)==18 and len(B_cols)==31

제외: ['AGE_BAND']
rows=307,511  결측=0  연체율=0.0807
A=18  B=31  C=15  M4=64


In [4]:
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5*(i+j-1)+1.0; i = j
    o = np.empty(N); o[J] = T; return o

def _fastdelong(preds, mpos):
    nt = preds.shape[1]; n = nt - mpos; k = preds.shape[0]
    pos, neg = preds[:, :mpos], preds[:, mpos:]
    tx = np.empty([k, mpos]); ty = np.empty([k, n]); tz = np.empty([k, nt])
    for r in range(k):
        tx[r] = _midrank(pos[r]); ty[r] = _midrank(neg[r]); tz[r] = _midrank(preds[r])
    aucs = tz[:, :mpos].sum(1)/mpos/n - (mpos+1.0)/2.0/n
    cov = np.cov((tz[:, :mpos]-tx)/n)/mpos + np.cov(1.0-(tz[:, mpos:]-ty)/mpos)/n
    return aucs, np.atleast_2d(cov)

def delong(y, s_full, s_reduced):
    y = np.asarray(y, float); order = (-y).argsort(kind="mergesort"); mpos = int(y.sum())
    preds = np.vstack((np.asarray(s_full)[order], np.asarray(s_reduced)[order]))
    aucs, cov = _fastdelong(preds, mpos)
    var = (np.array([[1., -1.]]) @ cov @ np.array([[1.], [-1.]])).item()
    d = aucs[0] - aucs[1]
    z, p = (0., 1.) if var <= 0 else (d/np.sqrt(var), 2*(1-stats.norm.cdf(abs(d/np.sqrt(var)))))
    return d, z, p

def boot_single(y, s, nb=None, seed=42):
    nb = nb or N_BOOT; rng = np.random.default_rng(seed)
    y = np.asarray(y); s = np.asarray(s); ip, ino = np.where(y==1)[0], np.where(y==0)[0]
    out = np.empty(nb)
    for i in range(nb):
        bi = np.concatenate([rng.choice(ip, ip.size, True), rng.choice(ino, ino.size, True)])
        out[i] = roc_auc_score(y[bi], s[bi])
    return tuple(np.percentile(out, [2.5, 97.5]))

def boot_diff(y, sf, sr, nb=None, seed=42):
    nb = nb or N_BOOT; rng = np.random.default_rng(seed)
    y = np.asarray(y); sf = np.asarray(sf); sr = np.asarray(sr)
    ip, ino = np.where(y==1)[0], np.where(y==0)[0]; out = np.empty(nb)
    for i in range(nb):
        bi = np.concatenate([rng.choice(ip, ip.size, True), rng.choice(ino, ino.size, True)])
        out[i] = roc_auc_score(y[bi], sf[bi]) - roc_auc_score(y[bi], sr[bi])
    return tuple(np.percentile(out, [2.5, 97.5]))

In [5]:
def oof(cols):
    X = m[cols].values
    skf = StratifiedKFold(5, shuffle=True, random_state=42)
    o = np.zeros(len(m))
    for tr, te in skf.split(X, y):
        pipe = Pipeline([("sc", StandardScaler()),
                         ("lr", LogisticRegression(C=1.0, max_iter=2000,
                                                   class_weight="balanced", solver="lbfgs"))])
        pipe.fit(X[tr], y[tr]); o[te] = pipe.predict_proba(X[te])[:, 1]
    return o

oof_M1 = oof(A_cols)
oof_M3 = oof(A_cols + B_cols)
oof_M4 = oof(A_cols + B_cols + C_cols)
AUC = {"M1": roc_auc_score(y, oof_M1), "M3": roc_auc_score(y, oof_M3), "M4": roc_auc_score(y, oof_M4)}
CI  = {k: boot_single(y, o) for k, o in [("M1", oof_M1), ("M3", oof_M3), ("M4", oof_M4)]}
model_tbl = pd.DataFrame([
    {"model": k, "X구성": x, "n_features": n, "OOF_AUC": round(AUC[k], 4),
     "CI_low": round(CI[k][0], 4), "CI_high": round(CI[k][1], 4)}
    for k, x, n in [("M1","A",len(A_cols)),
                    ("M3","A+B",len(A_cols+B_cols)),
                    ("M4","A+B+C",len(A_cols+B_cols+C_cols))]])
print(model_tbl.to_string(index=False))

model   X구성  n_features  OOF_AUC  CI_low  CI_high
   M1     A          18   0.6435  0.6400   0.6471
   M3   A+B          49   0.6797  0.6763   0.6830
   M4 A+B+C          64   0.6881  0.6850   0.6914


In [6]:
def compare(tag, sf, sr):
    d, z, p = delong(y, sf, sr); lo, hi = boot_diff(y, sf, sr)
    return {"비교": tag, "ΔAUC(%p)": round(d*100, 3), "DeLong_z": round(z, 2),
            "DeLong_p": f"{p:.2e}", "부트95%CI(%p)": f"[{lo*100:.2f}, {hi*100:.2f}]",
            "통계유의": p < 0.05, "실무유의(≥1%p)": d >= 0.01,
            "판정": "통과" if (p < 0.05 and d >= 0.01) else "미달"}

rq_df = pd.DataFrame([
    compare("M1 vs M4  (비금융 전체 B+C 기여, 요청 기준)", oof_M4, oof_M1),
    compare("M4 vs M3  (신규 C 순증 기여, RQ2 본질)",       oof_M4, oof_M3),
])
print(rq_df.to_string(index=False))

                              비교  ΔAUC(%p)  DeLong_z DeLong_p  부트95%CI(%p)  통계유의  실무유의(≥1%p) 판정
M1 vs M4  (비금융 전체 B+C 기여, 요청 기준)     4.467     34.20 0.00e+00 [4.21, 4.73]  True        True 통과
  M4 vs M3  (신규 C 순증 기여, RQ2 본질)     0.847     14.75 0.00e+00 [0.73, 0.96]  True       False 미달


In [7]:
import scipy.linalg

def independent_subset(cols, tol=1e-9):
    """완전 원핫(합=1) 등 완전공선 컬럼 자동 제거 → statsmodels 특이행렬 방지.
       제거되는 건 각 원핫 그룹의 기준범주 1개씩."""
    X  = m[cols].values.astype(float)
    Xs = np.column_stack([np.ones(len(X)), (X - X.mean(0))/(X.std(0)+1e-12)])
    _, r, piv = scipy.linalg.qr(Xs, mode="economic", pivoting=True)
    d = np.abs(np.diag(r)); rank = int((d > tol*d[0]).sum())
    keep    = [cols[i-1] for i in sorted(piv[:rank]) if i >= 1]
    dropped = [cols[i-1] for i in piv[rank:]        if i >= 1]
    return keep, dropped

keep_cols, dropped = independent_subset(A_cols + B_cols + C_cols)
if dropped: print("더미트랩 기준범주 제거:", dropped)

def coef_table(cols):
    binary = [x for x in cols if m[x].nunique() <= 2]
    Xstd = m[cols].astype(float).copy()
    for x in cols:                                   # 연속형만 표준화(더미는 0/1 → OR)
        if x not in binary and Xstd[x].std() > 0:
            Xstd[x] = (Xstd[x]-Xstd[x].mean())/Xstd[x].std()
    base = m[cols].astype(float)
    vif = pd.Series(np.diag(np.linalg.pinv(np.corrcoef(
        ((base-base.mean())/base.std()).values, rowvar=False))), index=cols)
    res = sm.Logit(y, sm.add_constant(Xstd)).fit(method="newton", maxiter=100, disp=0)
    star = lambda p: "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else ""
    t = pd.DataFrame({
        "variable": res.params.index, "group": [grp.get(x,"const") for x in res.params.index],
        "coef": res.params.values, "odds_ratio": np.exp(res.params.values),
        "p_value": res.pvalues.values, "sig": [star(p) for p in res.pvalues.values],
        "VIF": [vif.get(x, np.nan) for x in res.params.index]}).iloc[1:].reset_index(drop=True)
    t["_z"] = (t["coef"]/res.bse.values[1:]).abs()   # 로지스틱 변수중요도 ≈ |z|
    return t.sort_values("_z", ascending=False).drop(columns="_z").reset_index(drop=True)

coef_M4 = coef_table(keep_cols)   # 예측(CELL4)은 전체 변수, 해석은 기준범주 제외본
n_sig_C = int((coef_M4[coef_M4.group=="C"].sig != "").sum())
print(f"C(신규) 유의 변수: {n_sig_C} / {(coef_M4.group=='C').sum()}개")
print("C군 상위:", ", ".join(coef_M4[coef_M4.group=="C"].head(5)["variable"]))
print(coef_M4.head(15).assign(p_value=lambda d:d.p_value.map(lambda v:f"{v:.1e}")).to_string(index=False))

더미트랩 기준범주 제거: ['NAME_EDUCATION_TYPE_C_Secondary / secondary special', 'NAME_HOUSING_TYPE_C_House / apartment']
C(신규) 유의 변수: 10 / 13개
C군 상위: NAME_EDUCATION_TYPE_C_Higher education, FLAG_OWN_CAR, DEF_30_CNT_SOCIAL_CIRCLE, NAME_EDUCATION_TYPE_C_Incomplete higher, NAME_HOUSING_TYPE_C_Rented apartment
                               variable group      coef  odds_ratio  p_value sig       VIF
               BUREAU_ACTIVE_LOAN_COUNT     A  0.326845    1.386587 2.3e-195 ***  2.499240
                 BUREAU_NO_HISTORY_FLAG     A  0.612705    1.845417 5.6e-167 ***  1.300951
                      BUREAU_DEBT_RATIO     A  0.278268    1.320841 4.7e-141 ***  1.143344
                         YEARS_EMPLOYED     B -0.241923    0.785117 1.1e-129 ***  1.451319
                                    AGE     B -0.203388    0.815962  7.7e-99 ***  2.073562
 NAME_EDUCATION_TYPE_C_Higher education     C -0.403262    0.668137  5.3e-93 ***  1.267445
                        AMT_ANNUITY_LOG     A  1.448041    4.2547

In [8]:
with pd.ExcelWriter("M4_logistic_summary.xlsx", engine="openpyxl") as xw:
    model_tbl.to_excel(xw, sheet_name="01_모델비교", index=False)
    rq_df.to_excel(xw, sheet_name="02_RQ2판정", index=False)
    coef_M4.assign(p_value=coef_M4.p_value.map(lambda v:f"{v:.2e}")).to_excel(xw, sheet_name="03_M4변수중요도", index=False)

pd.DataFrame({"SK_ID_CURR": m["SK_ID_CURR"], "oof_M1": oof_M1, "oof_M3": oof_M3,
              "oof_M4": oof_M4, "TARGET": y}).to_csv("oof_M4_logistic.csv", index=False)

r1, r2 = rq_df.iloc[0], rq_df.iloc[1]
print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[신용평가] M4 = A+B+C (로지스틱) — RQ2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   M1 A       AUC {AUC['M1']:.4f}
   M3 A+B     AUC {AUC['M3']:.4f}
   M4 A+B+C   AUC {AUC['M4']:.4f}

■ M1 vs M4 (비금융 전체): ΔAUC {r1['ΔAUC(%p)']:+}%p, p={r1['DeLong_p']} → {r1['판정']}
■ M4 vs M3 (신규 C만):   ΔAUC {r2['ΔAUC(%p)']:+}%p, p={r2['DeLong_p']} → {r2['판정']}
   C {len(C_cols)}개 중 유의 {n_sig_C}개
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━""")

try:
    from google.colab import files
    files.download("M4_logistic_summary.xlsx")
    files.download("oof_M4_logistic.csv")
except Exception:
    print("saved.")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[신용평가] M4 = A+B+C (로지스틱) — RQ2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   M1 A       AUC 0.6435
   M3 A+B     AUC 0.6797
   M4 A+B+C   AUC 0.6881

■ M1 vs M4 (비금융 전체): ΔAUC +4.467%p, p=0.00e+00 → 통과
■ M4 vs M3 (신규 C만):   ΔAUC +0.847%p, p=0.00e+00 → 미달
   C 15개 중 유의 10개
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>